In [1]:
## Setup and Initialization

import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from the .env file
load_dotenv()

# Initialize the OpenAI client using the API key loaded from the environment
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Specify the embedding model to be used for text vectorization
model = "text-embedding-3-large" 

# Define a helper function to generate embeddings for a given text
def get_embedding(text, input_type="document"):
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

In [2]:
## Test Embeddings

# Test the embedding function to ensure it is working correctly
embed = get_embedding("RAG Technology")
len(embed)

3072

In [3]:
## Data Ingestion and Chunking

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF document from the provided URL
loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

# Initialize a text splitter to break the document into smaller, overlapping chunks
# This prevents loss of context between chunk boundaries
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(data)

# Display the resulting document chunks
documents

[Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue'),
 Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='NEW YORK, May 30, 2

In [4]:
## Prepare Documents

# Prepare the document chunks for database insertion by generating their embeddings
docs_to_insert = [
    {
        "text": doc.page_content,
        "embedding": get_embedding(doc.page_content)
    } for doc in documents
]

In [5]:
## Database Connection and Insertion

from pymongo import MongoClient

# Fetch the MongoDB connection URI from the environment variables
MONGO_URI = os.getenv("MONGO_URI")

# Connect to the MongoDB deployment
mongo_client = MongoClient(MONGO_URI)

# Access the specific database ("ragdb") and collection ("ragpdf")
collection = mongo_client["ragdb"]["ragpdf"]

# Insert the prepared documents (text + embeddings) into the MongoDB collection
result = collection.insert_many(docs_to_insert)
result

InsertManyResult([ObjectId('69cbf22264368ad5ddf483c7'), ObjectId('69cbf22264368ad5ddf483c8'), ObjectId('69cbf22264368ad5ddf483c9'), ObjectId('69cbf22264368ad5ddf483ca'), ObjectId('69cbf22264368ad5ddf483cb'), ObjectId('69cbf22264368ad5ddf483cc'), ObjectId('69cbf22264368ad5ddf483cd'), ObjectId('69cbf22264368ad5ddf483ce'), ObjectId('69cbf22264368ad5ddf483cf'), ObjectId('69cbf22264368ad5ddf483d0'), ObjectId('69cbf22264368ad5ddf483d1'), ObjectId('69cbf22264368ad5ddf483d2'), ObjectId('69cbf22264368ad5ddf483d3'), ObjectId('69cbf22264368ad5ddf483d4'), ObjectId('69cbf22264368ad5ddf483d5'), ObjectId('69cbf22264368ad5ddf483d6'), ObjectId('69cbf22264368ad5ddf483d7'), ObjectId('69cbf22264368ad5ddf483d8'), ObjectId('69cbf22264368ad5ddf483d9'), ObjectId('69cbf22264368ad5ddf483da'), ObjectId('69cbf22264368ad5ddf483db'), ObjectId('69cbf22264368ad5ddf483dc'), ObjectId('69cbf22264368ad5ddf483dd'), ObjectId('69cbf22264368ad5ddf483de'), ObjectId('69cbf22264368ad5ddf483df'), ObjectId('69cbf22264368ad5ddf483

In [6]:
## Create Vector Search Index

from pymongo.operations import SearchIndexModel
import time

# Create the vector search index model
index_name = "vector_index"
search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "numDimensions": 3072, # Matches the dimension size of the chosen OpenAI model
                "path": "embedding",
                "similarity": "cosine" # Cosine similarity measures the angle between vectors
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)

# Apply the index creation on the MongoDB collection
collection.create_search_index(model=search_index_model)

'vector_index'

In [7]:
## Poll for Index Readiness

# Wait for the initial sync of the vector index to complete
print("Polling to check if the index is ready. This may take up to a minute.")

predicate = None
if predicate is None:
    predicate = lambda index: index.get("queryable") is True
    
# Loop and check the status of the index every 5 seconds until it becomes queryable
while True:
    indices = list(collection.list_search_indexes(index_name))
    if len(indices) and predicate(indices[0]):
        break
    time.sleep(5)
    
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


In [8]:
## Query Embeddings

# Generate an embedding vector for an example user query
query_embeddings = get_embedding("AI Technology")
query_embeddings

[-0.014425646513700485,
 -0.010570360347628593,
 -0.019185928627848625,
 -0.025539005175232887,
 0.01002736296504736,
 0.023747112601995468,
 -0.013774048537015915,
 0.03223598003387451,
 -0.008416468277573586,
 0.046372026205062866,
 -0.010009262710809708,
 -0.028579793870449066,
 0.03534916788339615,
 0.01002736296504736,
 -0.03413647413253784,
 0.00032240504515357316,
 -0.00042082343134097755,
 -0.00952056422829628,
 -0.03140338137745857,
 -0.04546703025698662,
 -0.001997780054807663,
 0.006362126208841801,
 -0.027385197579860687,
 -0.012244604527950287,
 0.0051132310181856155,
 -0.01543924305588007,
 -0.0069639491848647594,
 -0.00011658902076305822,
 -0.0351138710975647,
 0.005262555554509163,
 0.03064318560063839,
 0.005620029289275408,
 0.03920445591211319,
 -0.015312543138861656,
 -0.009448165073990822,
 0.016561439260840416,
 0.018932528793811798,
 0.0281996950507164,
 0.001872211811132729,
 0.02765669673681259,
 0.030263086780905724,
 -0.01654333807528019,
 -0.0267879012972116

In [9]:
## Manual Vector Search

# Run a vector search query manually using MongoDB aggregation pipeline
results = collection.aggregate([
    {
        "$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": query_embeddings,
            "numCandidates": 3072,
            "limit": 5 # Return top 5 most relevant documents
        }
    }
])

# Process and print the matched documents
array_of_results = []
for doc in results:
    array_of_results.append(doc)
    
array_of_results

[{'_id': ObjectId('69cbf22264368ad5ddf483e6'),
  'text': 'artificial intelligence, in our offerings or partnerships; the growth and expansion of the market for database products and our ability to penetrate that\nmarket; our ability to integrate acquired businesses and technologies successfully or achieve the expected benefits of such acquisitions; our ability to',
  'embedding': [-0.02574683167040348,
   0.004299962893128395,
   -0.012185120023787022,
   0.0012026282493025064,
   0.03539811447262764,
   -0.02748648263514042,
   -0.015097144059836864,
   -0.02514173649251461,
   -0.021768325939774513,
   0.0384538471698761,
   0.02621578238904476,
   0.01310789119452238,
   0.029392534866929054,
   -0.0356098972260952,
   -0.058028701692819595,
   0.03439970314502716,
   -0.00804777629673481,
   -0.005899685434997082,
   -0.035488877445459366,
   -0.012775088660418987,
   0.002045980654656887,
   -0.02503584511578083,
   -0.005805139429867268,
   0.015550965443253517,
   -0.00501473248

In [10]:
## Reusable Search Function

# Define a helper function to streamline future vector search queries
def get_query_results(query):
    """Gets results from a vector search query."""
    
    # 1. Convert the plain text query into an embedding
    query_embedding = get_embedding(query, input_type="query")
    
    # 2. Build the aggregation pipeline to perform the vector search
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 3072,
                "limit": 5
            }
        }, 
        {
            # 3. Project only the text field, excluding the _id and embedding array for cleaner output
            "$project": {
                "_id": 0,
                "text": 1
            }
        }
    ]

    # Execute the pipeline against the collection
    results = collection.aggregate(pipeline)

    # Compile the results into a Python list
    array_of_results = []
    for doc in results:
        array_of_results.append(doc)
        
    return array_of_results

In [11]:
## Set the Query and Retrieve Context

# Define the user's question
query = "What are MongoDB's latest AI announcements?"

# Fetch the most relevant document chunks from MongoDB Vector Search
context_docs = get_query_results(query)

# Combine the retrieved document chunks into a single text string
context_string = " ".join([doc["text"] for doc in context_docs])

# Verify the combined context (Optional)
print("Retrieved Context Length:", len(context_string))

Retrieved Context Length: 1401


In [12]:
## Construct Prompt and Generate Answer via LLM

# Construct the final prompt containing both the retrieved context and the user's question
prompt = f"""Use the following pieces of context to answer the question at the end.
    
    Context:
    {context_string}
    
    Question: {query}
"""

# Define the OpenAI chat model you want to use
model_name = "gpt-4o"

# Generate the response using the securely initialized OpenAI client from Cell 1
completion = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# Extract and print the final generated answer
final_answer = completion.choices[0].message.content
print(final_answer)

MongoDB has announced the MongoDB AI Applications Program (MAAP), which is aimed at expanding its AI ecosystem. This program includes providing customers with reference architectures, pre-built partner integrations, and professional services to help quickly build AI-powered applications. Additionally, MongoDB has introduced significant updates with MongoDB 8.0, featuring performance improvements such as faster reads, updates, bulk inserts, and time series queries. They have also announced the general availability of Atlas Stream Processing for building sophisticated, event-driven applications with real-time data. Accenture has become the first global systems integrator to join MAAP and will establish a center of excellence focused on MongoDB projects.
